# Support Integrity Auditor (SIA) — Reproducible Pipeline

This notebook runs the whole pipeline end to end:

1. **Stage 1 — Pseudo-labels (self-supervised):** infer each ticket's true severity from independent signals and label priority mismatches *without any human-annotated labels*.
2. **Stage 2 — Classifier:** fine-tune DeBERTa-v3-small on those pseudo-labels.
3. **Stage 3 — Evidence Dossier:** for every flagged ticket, produce a structured, fully traceable explanation.

> Put the Kaggle CSV at `data/customer_support_tickets.csv` first. Run the notebook from the repo root.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd())=='notebooks' else os.getcwd())
import pandas as pd, json
pd.set_option('display.max_colwidth', 40)
DATA = 'data/customer_support_tickets.csv'   # change if needed
df = pd.read_csv(DATA)
print(df.shape); df.head(3)

## Stage 1 — Self-supervised pseudo-labels
We score severity from two independent signals (rule-based language + resolution time), fuse them, calibrate to the assigned-priority mix, then compare inferred vs assigned to label mismatches.

In [ ]:
from src.pseudo_label import generate_pseudo_labels, FusionConfig

labelled, diag, calib = generate_pseudo_labels(df, FusionConfig(mismatch_delta=2))
print(json.dumps(diag, indent=2, default=str))
labelled[['Ticket Subject','Ticket Priority','inferred_severity','severity_delta','mismatch','mismatch_type']].head(10)

### Why fuse? — signal ablation
Each signal alone vs fused. The weaker signal (usually resolution time) should carry the smaller fusion weight. This table goes in the README.

In [ ]:
for name, cfg in {
    'rule only': FusionConfig(signals=['rule'], weights={'rule':1.0}),
    'restime only': FusionConfig(signals=['restime'], weights={'restime':1.0}),
    'fused': FusionConfig(signals=['rule','restime'], weights={'rule':0.7,'restime':0.3}),
}.items():
    _, d, _ = generate_pseudo_labels(df, cfg)
    print(f"{name:14s} mismatch_rate={d['mismatch_rate']:.3f}  kappa={d['signal_agreement_kappa']}")

## Stage 2 — Fine-tune the classifier
Training needs a GPU, so run it as a script (Colab T4 is fine):
```bash
python train_pipeline.py --csv data/customer_support_tickets.csv --out artifacts --epochs 4
```
This saves the model, `calibration.json`, and `metrics.json` (with the pass/fail against the competition thresholds). Load the metrics back here:

In [ ]:
try:
    print(json.dumps(json.load(open('artifacts/metrics.json')), indent=2))
except FileNotFoundError:
    print('Train first (train_pipeline.py) to produce artifacts/metrics.json')

## Stage 3 — Evidence Dossiers
For every flagged ticket we build a dossier whose every evidence value is taken directly from a ticket field. The validator re-checks this and raises on anything untraceable, so hallucinated evidence cannot be emitted.

In [ ]:
from src.dossier import dossiers_for_df, validate_dossier
from src.signals import resolution_hours

dossiers = dossiers_for_df(labelled)            # uses the Stage 1 flag for the demo
print(f'{len(dossiers)} dossiers built')
print(json.dumps(dossiers[0], indent=2))

In [ ]:
# prove the no-hallucination guarantee
hrs = resolution_hours(labelled); idx_by_id = {str(r['Ticket ID']): i for i,(_,r) in enumerate(labelled.iterrows())}
ok = 0
for d in dossiers:
    row = labelled.set_index('Ticket ID').loc[type(labelled['Ticket ID'].iloc[0])(d['ticket_id'])].copy()
    h = hrs.loc[labelled.index[labelled['Ticket ID'].astype(str)==d['ticket_id']][0]]
    row['_restime_hours'] = float(h) if pd.notna(h) else None
    validate_dossier(d, row); ok += 1
print(f'{ok}/{len(dossiers)} dossiers traced clean — zero hallucinations')

## Inference on new tickets
Use the trained model + saved calibration on any CSV:
```bash
python predict.py --csv data/new_tickets.csv --model artifacts/model --calibration artifacts/calibration.json --out predictions
```
Or launch the web app:
```bash
streamlit run app.py
```